## Left paw + wheel wavelet clustering

Loads the super-session from `3.2_paw_wheel_subsample.ipynb`, fits PCA + KMeans, and propagates the clusters to every session (left paw + wheel wavelets merged on `Bin`).

In [ ]:
"""
IMPORTS
"""
import os
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

from segmentation_functions import idxs_from_files


In [ ]:
"""
PATHS AND FEATURES (proficient: left paw + wheel wavelets)
"""
base_path = '/home/ines/repositories/representation_learning_variability/paper-individuality/data/'
data_path = base_path + 'design_matrices/'
paw_wavelet_path = base_path + 'paw_wavelets/'
wheel_wavelet_path = base_path + 'wheel_wavelets/'

# must match 3.2_paw_wheel_subsample.ipynb (same columns, same order)
l_paw_cols = ['l_paw_x0.5', 'l_paw_x1.0', 'l_paw_x2.0', 'l_paw_x4.0', 'l_paw_x8.0',
              'l_paw_y0.5', 'l_paw_y1.0', 'l_paw_y2.0', 'l_paw_y4.0', 'l_paw_y8.0']
wheel_cols = ['avg_wheel_vel0.5', 'avg_wheel_vel1.0', 'avg_wheel_vel2.0', 'avg_wheel_vel4.0', 'avg_wheel_vel8.0']
var_interest = l_paw_cols + wheel_cols

super_name = 'session_zscored_supersession_wavelets_left_paw_wheel'
super_session = np.load(base_path + super_name)

state_name = 'left_paw_wheel_states'
states_path = base_path + 'left_paw_wheel_states_session_zscored/'
os.makedirs(states_path, exist_ok=True)

all_files = os.listdir(data_path)
design_matrices = [item for item in all_files if 'design_matrix' in item and 'standardized' not in item]
idxs, mouse_names = idxs_from_files(design_matrices)


In [ ]:
# Standardize the supersession and fit PCA
cutoff = 0.95
X_super = stats.zscore(super_session, axis=0)
pca_model = PCA(n_components=min(20, X_super.shape[1]))
X_pca = pca_model.fit_transform(X_super)
optimal_n_components = int(np.where(np.cumsum(pca_model.explained_variance_ratio_) >= cutoff)[0][0] + 1)
X_pca = X_pca[:, :optimal_n_components]
print('components for %.0f%% variance: %d' % (cutoff * 100, optimal_n_components))

plt.plot(np.cumsum(pca_model.explained_variance_ratio_))
plt.axhline(cutoff, color='r', ls='--')
plt.xlabel('Number of components'); plt.ylabel('Cumulative explained variance')
plt.show()


In [ ]:
# Choose k (elbow of the inertia curve)
inertia = []
K = range(1, 20)
for k in K:
    inertia.append(KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_pca).inertia_)
plt.plot(K, inertia, 'o-')
plt.xlabel('Number of clusters (k)'); plt.ylabel('Inertia')
plt.show()


In [ ]:
# Fit KMeans on the supersession
optimal_k = 4   # set from the inertia plot above

kmeans = KMeans(n_clusters=optimal_k, random_state=2024, n_init=10).fit(X_pca)
centroids = kmeans.cluster_centers_
cluster_values = kmeans.predict(X_pca)

global_mean = np.nanmean(X_super, axis=0)
global_std = np.nanstd(X_super, axis=0)


In [ ]:
# Propagate clusters to every session and save states
paw_files = os.listdir(paw_wavelet_path)
wheel_files = os.listdir(wheel_wavelet_path)
for m, mat in enumerate(idxs):

    mouse_name = mat[37:]
    session = mat[:36]
    if ('paw_vel_wavelets_' + session + '_' + mouse_name not in paw_files) or \
       ('wheel_vel_wavelets_' + session + '_' + mouse_name not in wheel_files):
        continue

    # left paw + wheel wavelets, merged on Bin (same samples)
    paw = pd.read_parquet(paw_wavelet_path + 'paw_vel_wavelets_' + session + '_' + mouse_name)
    wheel = pd.read_parquet(wheel_wavelet_path + 'wheel_vel_wavelets_' + session + '_' + mouse_name)
    design_matrix = paw[['Bin'] + l_paw_cols].merge(wheel[['Bin'] + wheel_cols], on='Bin', how='inner')

    var_array = design_matrix[var_interest].to_numpy()
    not_nan = ~np.isnan(var_array).any(axis=1)

    mouse_data = stats.zscore(var_array[not_nan], axis=0, nan_policy='omit')  # within session
    mouse_data = (mouse_data - global_mean) / global_std                      # global
    session_pca = pca_model.transform(mouse_data)[:, :optimal_n_components]
    states = np.argmin(cdist(session_pca, centroids, metric='euclidean'), axis=1)

    filename = states_path + 'most_likely_states_' + str(optimal_k) + '_' + mouse_name + session
    np.save(filename, np.array([states, np.array(design_matrix['Bin'][not_nan])]))


In [ ]:
# Cluster means across features
all_data = pd.DataFrame(X_super, columns=var_interest)
all_data[state_name] = cluster_values
wavelet_df = all_data.melt(id_vars=[state_name], value_vars=var_interest)

plt.figure(figsize=(11, 4))
sns.barplot(data=wavelet_df, x='variable', y='value', hue=state_name,
            palette=sns.color_palette('Set2', optimal_k), alpha=.8)
plt.xticks(rotation=90); plt.ylabel('z-scored power'); plt.title('Cluster means (left paw + wheel)')
plt.show()
